# 08 - Deploy the Workspace Owner Agent

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Creates or updates and publishes a **separate owner Agent**, grounded only on the secured Workspace Owner semantic model. It does not modify the model, grant permissions or change the central Agent or app.

**Before running:** configure the owner model's fixed-identity connection with source SSO disabled, run an owner-enabled review through Gold, then complete **07_OwnerAccessSync**. Setup stamps the exact owner model ID below. Do not substitute the governance model.

**Run order:** FOS setup publishes the SDK in **FAR_Runtime** and binds this notebook to it. Run all cells after Gold and successful owner-access sync. This standalone notebook is not a pipeline activity.

**Runtime:** `OWNER_AGENT_RUNTIME` defaults to `"standard"`. Select `"preview"` with a separately named test agent to try the preview runtime. Publishing captures the setting; owner-model verification and access requirements are unchanged.

Model RLS and each caller's permissions enforce workspace access. Instructions and table selection are not security boundaries. After publication, approve only controlled test readers: query-only Agent access, owner-model Read and WorkspaceOwner membership. Build, FAR workspace roles and raw Lakehouse/SQL access are not required. Test actual users, denied workspaces, expiry and revocation before broader sharing. Maintain daily 07 sync.

See the [owner Agent guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-agent.md).


In [ ]:
from importlib.metadata import version
if version("fabric-data-agent-sdk") != "0.1.30a0":
    raise RuntimeError("Rerun FOS_Install_FAR to publish the SDK in FAR_Runtime, then restart this notebook session with that Environment.")


In [ ]:
WORKSPACE_ID = ""
OWNER_SEMANTIC_MODEL_ID = ""
OWNER_AGENT_NAME = "Fabric Arch Review - Workspace Owner Agent"
OWNER_AGENT_RUNTIME = "standard"


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))

from uuid import UUID

UUID(WORKSPACE_ID)
UUID(OWNER_SEMANTIC_MODEL_ID)
if not isinstance(OWNER_AGENT_NAME, str) or not OWNER_AGENT_NAME.strip():
    raise ValueError("OWNER_AGENT_NAME must be a nonempty string.")

from orchestration.fabric_api import FabricClient
from reports.agent.owner_agent import deploy_owner_agent

_version = Path(REPO_DIR, "VERSION").read_text(encoding="utf-8-sig").strip()
_agent_id = deploy_owner_agent(
    client=FabricClient(lambda: notebookutils.credentials.getToken("pbi")),
    workspace_id=WORKSPACE_ID, model_id=OWNER_SEMANTIC_MODEL_ID,
    agent_name=OWNER_AGENT_NAME, version=_version,
    runtime=OWNER_AGENT_RUNTIME,
)
from orchestration.folders import organize
_folder_client = FabricClient(lambda: notebookutils.credentials.getToken('pbi'))
organize(_folder_client, WORKSPACE_ID, [(_agent_id, 'DataAgent', 'FAR/Workspace Owner')])
_ctx = notebookutils.runtime.context
organize(_folder_client, _ctx.get('currentWorkspaceId') or _ctx.get('workspaceId'),
         [(_ctx['currentNotebookId'], 'Notebook', 'FAR/Workspace Owner')])
print("Workspace Owner Agent published and source verified:", _agent_id)
print("No permissions were granted. Complete actual-reader acceptance before broader sharing; keep daily 07 access sync active.")
